In [1]:
!pip install transformers peft datasets torch accelerate PyYAML tqdm scikit-learn -q

In [2]:
!pip install torchao --upgrade -q

In [3]:
import transformers, peft, datasets, torch

print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("datasets    :", datasets.__version__)
print("torch       :", torch.__version__)
print("GPU         :", torch.cuda.is_available())

transformers: 5.12.0
peft        : 0.19.1
datasets    : 4.0.0
torch       : 2.11.0+cu128
GPU         : True


In [4]:
import os, yaml

folders = ["configs", "data", "model", "training", "inference", "utils", "outputs", "logs"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

config_content = {
    "model": {
        "base_model": "gpt2",
        "max_length": 128
    },
    "lora": {
        "r": 8,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
        "target_modules": ["c_attn"],
        "bias": "none"
    },
    "training": {
        "output_dir": "./outputs",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 8,
        "per_device_eval_batch_size": 8,
        "learning_rate": 2.0e-4,
        "warmup_steps": 100,
        "logging_steps": 50,
        "save_steps": 200,
        "eval_steps": 200,
        "weight_decay": 0.01,
        "fp16": True,
        "gradient_accumulation_steps": 1,
        "seed": 42
    },
    "data": {
        "label_map": {
            0: "negative",
            1: "positive",
            2: "neutral"
        }
    },
    "inference": {
        "max_new_tokens": 10,
        "temperature": 0.1,
        "top_p": 0.9,
        "top_k": 50,
        "repetition_penalty": 1.3,
        "num_return_sequences": 1
    }
}

with open("configs/config.yaml", "w") as f:
    yaml.dump(config_content, f)

with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

print(" Folders created!")
print("config.yaml saved!")
print(f"Label map : {cfg['data']['label_map']}")
print(f"LoRA rank : {cfg['lora']['r']}")
print(f"Epochs    : {cfg['training']['num_train_epochs']}")

 Folders created!
config.yaml saved!
Label map : {0: 'negative', 1: 'positive', 2: 'neutral'}
LoRA rank : 8
Epochs    : 3


In [5]:
import sys
sys.path.insert(0, '/content')

data_loader_code = '''
from datasets import load_dataset
from transformers import GPT2Tokenizer
from torch.utils.data import Dataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import torch
import numpy as np
import yaml
import logging
from collections import Counter

logger = logging.getLogger(__name__)


def load_config(config_path="configs/config.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)


class FinanceDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids      = encodings["input_ids"]
        self.attention_mask = encodings["attention_mask"]
        self.labels_list    = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids"     : torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels"        : torch.tensor(self.input_ids[idx], dtype=torch.long),
        }


def get_tokenizer(model_name="gpt2"):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def convert_to_prompt_format(texts, labels, label_map):
    """
    Option C format:
    "Sentence: <text> Sentiment: <positive/negative/neutral>"
    """
    prompts = []
    for text, label in zip(texts, labels):
        sentiment = label_map[int(label)]
        prompt    = f"Sentence: {text.strip()} Sentiment: {sentiment}"
        prompts.append(prompt)
    return prompts


def get_weighted_sampler(dataset):
    """
    Weighted Random Sampler:
    Negative (15%) → High weight → zyada chances
    Positive (20%) → Medium weight
    Neutral  (65%) → Low weight → kam chances

    Result: Model teeno classes equally dekhta hai!
    """
    labels        = dataset.labels_list
    class_counts  = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[label] for label in labels]
    sample_weights = torch.tensor(sample_weights, dtype=torch.float)

    sampler = WeightedRandomSampler(
        weights     = sample_weights,
        num_samples = len(sample_weights),
        replacement = True,
    )
    return sampler


def prepare_datasets(config_path="configs/config.yaml"):
    config    = load_config(config_path)
    model_cfg = config["model"]

    label_map = {
        0: "negative",
        1: "positive",
        2: "neutral",
    }

    logger.info("Loading dataset...")
    raw_train = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train")
    raw_val   = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")

    # FIX: Convert to list before combining
    all_texts  = list(raw_train["text"])  + list(raw_val["text"])
    all_labels = list(raw_train["label"]) + list(raw_val["label"])

    logger.info(f"Total samples: {len(all_texts)}")

    # ── STRATIFIED SPLIT ──────────────────────────────────
    train_texts, temp_texts, train_labels, temp_labels = train_test_split(
        all_texts, all_labels,
        test_size    = 0.20,
        random_state = 42,
        stratify     = all_labels,
    )

    val_texts, test_texts, val_labels, test_labels = train_test_split(
        temp_texts, temp_labels,
        test_size    = 0.50,
        random_state = 42,
        stratify     = temp_labels,
    )

    logger.info(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

    # ── VERIFY STRATIFICATION ─────────────────────────────
    for split_name, split_labels in [
        ("Train", train_labels),
        ("Val",   val_labels),
        ("Test",  test_labels)
    ]:
        counts = Counter(split_labels)
        total  = len(split_labels)
        print(
            f"{split_name} → "
            f"Neg: {counts[0]/total*100:.1f}% | "
            f"Pos: {counts[1]/total*100:.1f}% | "
            f"Neu: {counts[2]/total*100:.1f}%"
        )

    # ── PROMPT FORMAT ─────────────────────────────────────
    train_prompts = convert_to_prompt_format(train_texts, train_labels, label_map)
    val_prompts   = convert_to_prompt_format(val_texts,   val_labels,   label_map)
    test_prompts  = convert_to_prompt_format(test_texts,  test_labels,  label_map)

    # ── TOKENIZE ──────────────────────────────────────────
    tokenizer = get_tokenizer(model_cfg["base_model"])

    def tokenize(text_list):
        return tokenizer(
            text_list,
            truncation     = True,
            padding        = "max_length",
            max_length     = model_cfg["max_length"],
            return_tensors = None,
        )

    train_dataset = FinanceDataset(tokenize(train_prompts), train_labels)
    val_dataset   = FinanceDataset(tokenize(val_prompts),   val_labels)
    test_dataset  = FinanceDataset(tokenize(test_prompts),  test_labels)

    # ── WEIGHTED SAMPLER — train only ─────────────────────
    train_sampler = get_weighted_sampler(train_dataset)

    return train_dataset, val_dataset, test_dataset, tokenizer, test_prompts, test_labels, label_map, train_sampler
'''

with open("data/data_loader.py", "w") as f:
    f.write(data_loader_code)

print("data/data_loader.py saved!")

data/data_loader.py saved!


In [6]:
from data.data_loader import prepare_datasets

train, val, test, tokenizer, test_texts, test_labels, label_map, train_sampler = prepare_datasets()

print(f"\nTrain  : {len(train)}")
print(f"Val    : {len(val)}")
print(f"Test   : {len(test)}")
print(f"\nSample prompt: {test_texts[0]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train → Neg: 15.0% | Pos: 20.1% | Neu: 64.9%
Val → Neg: 15.0% | Pos: 20.1% | Neu: 64.9%
Test → Neg: 15.0% | Pos: 20.1% | Neu: 64.9%

Train  : 9544
Val    : 1193
Test   : 1194

Sample prompt: Sentence: $MTSC - MTS Systems declares $0.30 dividend https://t.co/QRVl05IT61 Sentiment: neutral


In [7]:
model_setup_code = '''
from transformers import GPT2LMHeadModel
from peft import get_peft_model, LoraConfig, TaskType
import yaml
import logging

logger = logging.getLogger(__name__)


def load_config(config_path="configs/config.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)


def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params : {trainable:,}")
    print(f"Total params     : {total:,}")
    print(f"Trainable %      : {100 * trainable / total:.4f}%")


def build_lora_model(config_path="configs/config.yaml"):
    config    = load_config(config_path)
    model_cfg = config["model"]
    lora_cfg  = config["lora"]

    base_model  = GPT2LMHeadModel.from_pretrained(model_cfg["base_model"])

    lora_config = LoraConfig(
        task_type      = TaskType.CAUSAL_LM,
        r              = lora_cfg["r"],
        lora_alpha     = lora_cfg["lora_alpha"],
        lora_dropout   = lora_cfg["lora_dropout"],
        target_modules = lora_cfg["target_modules"],
        bias           = lora_cfg["bias"],
    )

    model = get_peft_model(base_model, lora_config)
    return model
'''

with open("model/model_setup.py", "w") as f:
    f.write(model_setup_code)

print(" model/model_setup.py saved!")

 model/model_setup.py saved!


In [8]:
from model.model_setup import build_lora_model, print_trainable_parameters

model = build_lora_model()
print_trainable_parameters(model)
print(" LoRA model ready!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Trainable params : 294,912
Total params     : 124,734,720
Trainable %      : 0.2364%
 LoRA model ready!


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import yaml, os, torch, math

with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

train_cfg = config["training"]
use_fp16  = train_cfg["fp16"] and torch.cuda.is_available()

os.makedirs(train_cfg["output_dir"], exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = train_cfg["output_dir"],
    num_train_epochs            = train_cfg["num_train_epochs"],
    per_device_train_batch_size = train_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size  = train_cfg["per_device_eval_batch_size"],
    learning_rate               = train_cfg["learning_rate"],
    warmup_steps                = train_cfg["warmup_steps"],
    weight_decay                = train_cfg["weight_decay"],
    logging_steps               = train_cfg["logging_steps"],
    save_steps                  = train_cfg["save_steps"],
    eval_strategy               = "steps",
    save_strategy               = "steps",
    load_best_model_at_end      = True,
    fp16                        = use_fp16,
    gradient_accumulation_steps = train_cfg["gradient_accumulation_steps"],
    seed                        = train_cfg["seed"],
    report_to                   = "none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False,
)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train,
    eval_dataset     = val,
    processing_class = tokenizer,
    data_collator    = data_collator,
)

print(" Starting training... (10-15 min ☕)")
train_result = trainer.train()

final_path = os.path.join(train_cfg["output_dir"], "final_model")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

eval_results = trainer.evaluate()
loss         = eval_results["eval_loss"]
perplexity   = math.exp(loss)

print("\n" + "="*50)
print("TRAINING COMPLETE!")
print("="*50)
print(f"Training Loss : {train_result.training_loss:.4f}")
print(f"Val Loss      : {loss:.4f}")
print(f"Perplexity    : {perplexity:.2f}")
print("="*50)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


 Starting training... (10-15 min ☕)


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
50,5.344701,4.811334
100,4.447528,3.811744
150,3.926942,3.667908
200,3.805728,3.617765
250,3.807603,3.589189
300,3.729849,3.541470
350,3.722778,3.522903
400,3.718503,3.524717
450,3.634424,3.476404
500,3.619795,3.452755


Step,Training Loss,Validation Loss
50,5.344701,4.811334
100,4.447528,3.811744
150,3.926942,3.667908
200,3.805728,3.617765
250,3.807603,3.589189
300,3.729849,3.541470
350,3.722778,3.522903
400,3.718503,3.524717
450,3.634424,3.476404
500,3.619795,3.452755


Training Loss,Validation Loss,Step
3.427370,3.250031,3579



TRAINING COMPLETE!
Training Loss : 3.5602
Val Loss      : 3.2500
Perplexity    : 25.79


In [10]:
import torch, math
from transformers import GPT2LMHeadModel
from peft import PeftModel, PeftConfig
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate_sentiment_token_loss(model, test_texts, test_labels, tokenizer, label_map, device, model_name="Model"):
    model.eval()
    model.to(device)

    total_loss       = 0
    correct          = 0
    total            = 0
    sentiment_tokens = {v: tokenizer.encode(" " + v)[0] for v in label_map.values()}

    for text, true_label in tqdm(zip(test_texts, test_labels), total=len(test_texts), desc=model_name):
        prompt    = text.rsplit(" ", 1)[0] + " "
        true_word = label_map[int(true_label)]
        true_tok  = sentiment_tokens[true_word]

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=120).to(device)

        with torch.no_grad():
            outputs  = model(**inputs)
            logits   = outputs.logits[0, -1, :]

            log_probs  = torch.nn.functional.log_softmax(logits, dim=-1)
            total_loss += -log_probs[true_tok].item()

            sent_ids       = list(sentiment_tokens.values())
            sent_logits    = logits[sent_ids]
            predicted_idx  = sent_logits.argmax().item()
            predicted_word = list(sentiment_tokens.keys())[predicted_idx]

            correct += int(predicted_word == true_word)
            total   += 1

    avg_loss   = total_loss / total
    perplexity = math.exp(min(avg_loss, 300))
    accuracy   = (correct / total) * 100
    return round(avg_loss, 4), round(perplexity, 2), round(accuracy, 2)


# ── BASE GPT-2 ────────────────────────────────────────────
print("="*55)
print("  BASE GPT-2 (Before Fine-Tuning)")
print("="*55)
base_model                    = GPT2LMHeadModel.from_pretrained("gpt2")
base_loss, base_ppl, base_acc = evaluate_sentiment_token_loss(
    base_model, test_texts, test_labels, tokenizer, label_map, device, "Base GPT-2"
)
print(f"Loss       : {base_loss}")
print(f"Perplexity : {base_ppl}")
print(f"Accuracy   : {base_acc}%")
del base_model
torch.cuda.empty_cache()


# ── FINE-TUNED LoRA ───────────────────────────────────────
print("\n" + "="*55)
print("  FINE-TUNED LoRA (After Fine-Tuning)")
print("="*55)
peft_config               = PeftConfig.from_pretrained("./outputs/final_model")
ft_base                   = GPT2LMHeadModel.from_pretrained(peft_config.base_model_name_or_path)
ft_model                  = PeftModel.from_pretrained(ft_base, "./outputs/final_model")
ft_model                  = ft_model.merge_and_unload()
ft_loss, ft_ppl, ft_acc   = evaluate_sentiment_token_loss(
    ft_model, test_texts, test_labels, tokenizer, label_map, device, "Fine-Tuned LoRA"
)
print(f"Loss       : {ft_loss}")
print(f"Perplexity : {ft_ppl}")
print(f"Accuracy   : {ft_acc}%")


# ── FINAL REPORT ──────────────────────────────────────────
print("\n")
print("╔══════════════════════════════════════════════════════╗")
print("║         BEFORE vs AFTER — FINAL REPORT              ║")
print("╠══════════════════════════════╦═════════════╦════════╣")
print("║  Metric                      ║  Base GPT-2 ║  LoRA  ║")
print("╠══════════════════════════════╬═════════════╬════════╣")
print(f"║  Sentiment Loss         ↓    ║  {base_loss:<11} ║  {ft_loss:<6} ║")
print(f"║  Sentiment Perplexity   ↓    ║  {base_ppl:<11} ║  {ft_ppl:<6} ║")
print(f"║  Sentiment Accuracy %   ↑    ║  {base_acc:<11} ║  {ft_acc:<6} ║")
print("╠══════════════════════════════╩═════════════╩════════╣")
print(f"║  Accuracy Gain : {base_acc}% → {ft_acc}%                    ║")
print(f"║  Loss Drop     : {base_loss} → {ft_loss}                      ║")
print("╚══════════════════════════════════════════════════════╝")

  BASE GPT-2 (Before Fine-Tuning)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Base GPT-2: 100%|██████████| 1194/1194 [00:15<00:00, 76.83it/s]


Loss       : 14.97
Perplexity : 3172524.44
Accuracy   : 20.52%

  FINE-TUNED LoRA (After Fine-Tuning)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Fine-Tuned LoRA: 100%|██████████| 1194/1194 [00:14<00:00, 79.74it/s]

Loss       : 5.8536
Perplexity : 348.5
Accuracy   : 65.75%


╔══════════════════════════════════════════════════════╗
║         BEFORE vs AFTER — FINAL REPORT              ║
╠══════════════════════════════╦═════════════╦════════╣
║  Metric                      ║  Base GPT-2 ║  LoRA  ║
╠══════════════════════════════╬═════════════╬════════╣
║  Sentiment Loss         ↓    ║  14.97       ║  5.8536 ║
║  Sentiment Perplexity   ↓    ║  3172524.44  ║  348.5  ║
║  Sentiment Accuracy %   ↑    ║  20.52       ║  65.75  ║
╠══════════════════════════════╩═════════════╩════════╣
║  Accuracy Gain : 20.52% → 65.75%                    ║
║  Loss Drop     : 14.97 → 5.8536                      ║
╚══════════════════════════════════════════════════════╝
